# 🚀 Improved OpenUSD Hero Shot for Blender
**Larger scale, better geometry, stronger lighting — perfect for Blender import**

In [ ]:
!pip install usd-core --quiet

from pxr import Usd, UsdGeom, UsdLux, UsdShade, Sdf, Gf
from google.colab import files

print("✅ Libraries loaded")

In [ ]:
stage = Usd.Stage.CreateNew("better_file.usd")

# Hero Lantern
hero = UsdGeom.Xform.Define(stage, "/World/Hero/Lantern")
Usd.ModelAPI(hero.GetPrim()).SetKind("component")

body = UsdGeom.Mesh.Define(stage, "/World/Hero/Lantern/Body")
body.CreatePointsAttr().Set([
    (-2,-2,0), (2,-2,0), (2,2,0), (-2,2,0),
    (-2.5,-2.5,6), (2.5,-2.5,6), (2.5,2.5,6), (-2.5,2.5,6)
])
body.CreateFaceVertexCountsAttr().Set([4,4,4,4,4,4])
body.CreateFaceVertexIndicesAttr().Set([0,1,5,4, 1,2,6,5, 2,3,7,6, 3,0,4,7, 4,5,6,7, 0,3,2,1])

# Primvar
pv = UsdGeom.PrimvarsAPI(body).CreatePrimvar("displayColor", Sdf.ValueTypeNames.Color3fArray, UsdGeom.Tokens.vertex)
pv.Set([(0.9, 0.55, 0.2)] * 8)

# Material
mat = UsdShade.Material.Define(stage, "/World/Materials/LanternMetal")
shader = UsdShade.Shader.Define(stage, "/World/Materials/LanternMetal/Preview")
shader.CreateIdAttr("UsdPreviewSurface")
shader.CreateInput("diffuseColor", Sdf.ValueTypeNames.Color3f).Set((0.85, 0.75, 0.55))
shader.CreateInput("metallic", Sdf.ValueTypeNames.Float).Set(0.9)
UsdShade.MaterialBindingAPI.Apply(hero.GetPrim()).Bind(mat)

print("✅ Lantern created (larger & better)")

In [ ]:
# Strong lighting
key = UsdLux.SphereLight.Define(stage, "/World/Lights/KeyLight")
key.CreateIntensityAttr().Set(120000)
key.CreateRadiusAttr().Set(5)
key.AddTranslateOp().Set(Gf.Vec3d(15, 20, 18))

fill = UsdLux.SphereLight.Define(stage, "/World/Lights/FillLight")
fill.CreateIntensityAttr().Set(40000)
fill.CreateRadiusAttr().Set(6)
fill.AddTranslateOp().Set(Gf.Vec3d(-18, 10, -12))

print("✅ Strong lighting added")

In [ ]:
# Camera
cam = UsdGeom.Camera.Define(stage, "/World/Camera/ShotCamera")
cam.CreateFocalLengthAttr().Set(50)
xform = UsdGeom.Xformable(cam.GetPrim())
xform.ClearXformOpOrder()
xform.AddTranslateOp().Set(Gf.Vec3d(18, 14, 22))
xform.AddRotateXYZOp().Set(Gf.Vec3f(-30, 40, 0))

print("✅ Camera set")

In [ ]:
# Large ground
ground = UsdGeom.Mesh.Define(stage, "/World/Ground")
ground.CreatePointsAttr().Set([(-100,0,-100),(100,0,-100),(100,0,100),(-100,0,100)])
ground.CreateFaceVertexCountsAttr().Set([4])
ground.CreateFaceVertexIndicesAttr().Set([0,1,2,3])

# Trees using instancing
proto = UsdGeom.Mesh.Define(stage, "/World/Prototypes/Tree")
proto.CreatePointsAttr().Set([(0,0,0), (0,8,0), (3,5,0), (-3,5,0)])
proto.CreateFaceVertexCountsAttr().Set([3,3])
proto.CreateFaceVertexIndicesAttr().Set([0,1,2, 0,1,3])

for i in range(15):
    tree = UsdGeom.Xform.Define(stage, f"/World/Trees/Tree_{i:02d}")
    prim = tree.GetPrim()
    prim.GetReferences().AddReference(stage.GetRootLayer().identifier, "/World/Prototypes/Tree")
    prim.SetInstanceable(True)
    tree.AddTranslateOp().Set(Gf.Vec3d(i*8-55, 0, (i%6)*10-20))

print("✅ Large ground + visible trees created")

In [ ]:
stage.Save()
print("\n✅ better_file.usd successfully created!")
files.download("better_file.usd")